In [ ]:
%cd /Users/aliceliusyncrowin/Projects/tata-schedule-fetch
import json
import pandas as pd
import psycopg2
import os
import datetime as dt

with open("local.settings.json", "r") as f:
    settings = json.load(f)["Values"]

for key, value in settings.items():
    os.environ[key] = value

database = os.getenv("POSTGRES_DB")
user = os.getenv("POSTGRES_USER")
password = os.getenv("POSTGRES_PASSWORD")
host = os.getenv("POSTGRES_HOST")

In [ ]:
conn = psycopg2.connect(
    dbname=database,
    user=user,
    host=host,
    password=password,
    port="5432",
)
start_date = dt.date(2026, 6, 11)
end_date = dt.date(2026, 6,21)
# get latest updated data from each day after 6-11-2026 inclusive
with conn.cursor() as cur:
    cur.execute("""
        SELECT DISTINCT ON (schedule_date_ist)
                schedule_date_ist, time_retrieved, time_retrieved_ist, schedule_data
                FROM tata_schedule
                WHERE schedule_date_ist BETWEEN %s AND %s
                ORDER BY schedule_date_ist, time_retrieved DESC;
                """,
                
        (start_date, end_date))
    
    rows = cur.fetchall()
conn.close()    
rows



In [ ]:
class NetSchdDataList(object):
    def __init__(self, ASTypeName: str, NetSchdAmount: list[float], PXExchangeTypeName: str, PXTransactionTypeName: str, EnergyScheduleTypeName: str, EnergyScheduleSubTypeName: str, DataIntegrationTimingTypeName: str):
        self.ASTypeName = ASTypeName
        self.NetSchdAmount = NetSchdAmount
        self.PXExchangeTypeName = PXExchangeTypeName
        self.PXTransactionTypeName = PXTransactionTypeName
        self.EnergyScheduleTypeName = EnergyScheduleTypeName
        self.EnergyScheduleSubTypeName = EnergyScheduleSubTypeName
        self.DataIntegrationTimingTypeName = DataIntegrationTimingTypeName
class REMCDeclarationData(object):
    def __init__(self, StationSchAmount: list[float], AvlCapacityAmount: list[float], RLDCForecastAmount: list[float]):
        self.StationSchAmount = StationSchAmount
        self.AvlCapacityAmount = AvlCapacityAmount
        self.RLDCForecastAmount = RLDCForecastAmount
class REMCDeclarationList(object):
    def __init__(self, CreatedOn: str, revisionNo: int, UnitAcronym: str, UtilAcronym: str, IsUnitWiseData: int, GeneratorTypeName: str, REMCInpRevisionNo: int, REMCDeclarationData: REMCDeclarationData, GeneratorSubTypeName: int):
        self.CreatedOn = CreatedOn
        self.revisionNo = revisionNo
        self.UnitAcronym = UnitAcronym
        self.UtilAcronym = UtilAcronym
        self.IsUnitWiseData = IsUnitWiseData
        self.GeneratorTypeName = GeneratorTypeName
        self.REMCInpRevisionNo = REMCInpRevisionNo
        self.REMCDeclarationData = REMCDeclarationData
        self.GeneratorSubTypeName = GeneratorSubTypeName
class NetSchedulesSummary(object):
    def __init__(self, net_schd_data_list: list[dict], TotalNetSchdAmount: list[float]):
        self.NetSchdDataList = [
            NetSchdDataList(
                ASTypeName=item.get("ASTypeName"),
                NetSchdAmount=item.get("NetSchdAmount"),
                PXExchangeTypeName=item.get("PXExchangeTypeName"),
                PXTransactionTypeName=item.get("PXTransactionTypeName"),
                EnergyScheduleTypeName=item.get("EnergyScheduleTypeName"),
                EnergyScheduleSubTypeName=item.get("EnergyScheduleSubTypeName"),
                DataIntegrationTimingTypeName=item.get("DataIntegrationTimingTypeName"),
            )
            for item in (net_schd_data_list or [])
        ]
        self.TotalNetSchdAmount = TotalNetSchdAmount

def get_net_schedule_summary(dict_schedule_response_body: dict) -> NetSchedulesSummary | None:
    target = dict_schedule_response_body.get("GroupWiseDataList", [{}])[0].get("NetScheduleSummary")
    if not target:
        return None
    return NetSchedulesSummary(
        net_schd_data_list=target.get("NetSchdDataList"),
        TotalNetSchdAmount=target.get("TotalNetSchdAmount"),
    )

def get_remc_declaration_list(dict_schedule_response_body: dict) -> REMCDeclarationList | None:
    target_dict = dict_schedule_response_body.get("GroupWiseDataList", [{}])[0].get("REMCDeclarationList", [{}])[0]
    if target_dict is not None:
        print(target_dict)
        return REMCDeclarationList(
            CreatedOn=target_dict.get("CreateOdn"),
            revisionNo=target_dict.get("revisionNo"),
            UnitAcronym=target_dict.get("UnitAcronym"),
            UtilAcronym=target_dict.get("UtilAcronym"),
            IsUnitWiseData=target_dict.get("IsUnitWiseData"),
            GeneratorTypeName=target_dict.get("GeneratorTypeName"),
            REMCInpRevisionNo=target_dict.get("REMCInpRevisionNo"),
            REMCDeclarationData=REMCDeclarationData(
                StationSchAmount=target_dict.get("REMCDeclarationData", {}).get("StationSchAmount"),
                AvlCapacityAmount=target_dict.get("REMCDeclarationData", {}).get("AvlCapacityAmount"),
                RLDCForecastAmount=target_dict.get("REMCDeclarationData", {}).get("RLDCForecastAmount")
            ),
            GeneratorSubTypeName=target_dict.get("GeneratorSubTypeName")
        )
        
    else:
        print("REMCDeclarationList not found in the response body.")
        return None
    
class ResponseBody(object):
    def __init__(self, FullSchdRevisionNo: int, SchedulePublishedTime: str, response_body_dict: dict):
        
        self.FullSchdRevisionNo = FullSchdRevisionNo
        self.SchedulePublishedTime = SchedulePublishedTime
        self.NetScheduleSummary = get_net_schedule_summary(response_body_dict)  
        self.REMCDeclarationList = get_remc_declaration_list(response_body_dict)



In [ ]:
# Create object for each row 
scheduled_data_objects = []
response_body_objects = []
for row in rows:
    schedule_date_ist, time_retrieved, time_retrieved_ist, data = row
    scheduled_data_objects.append({
        "schedule_date_ist": schedule_date_ist,
        "time_retrieved": time_retrieved,
        "time_retrieved_ist": time_retrieved_ist,
        "schedule_data": data
    })
    # Create one station's data
    response_body = ResponseBody(
        FullSchdRevisionNo=data['data']['ResponseBody']['FullSchdRevisionNo'],
        SchedulePublishedTime=data['data']['ResponseBody']['SchedulePublishedTime'],
        response_body_dict=data['data']['ResponseBody'],
    )
    response_body_objects.append(response_body)

In [ ]:
os_remc_list = []
as_emergency = []
as_shortfall = []
total_net_schd_amount = []
station_sched_amount = []

for response_body in response_body_objects:
    date = response_body.SchedulePublishedTime
    for net_schd_data_item in response_body.NetScheduleSummary.NetSchdDataList:
        if net_schd_data_item.ASTypeName == "EMERGENCY":
            as_emergency.append((date,net_schd_data_item))
        elif net_schd_data_item.ASTypeName == "SHORTFALL":
            as_shortfall.append((date,net_schd_data_item))
        else:
            cond = (
                net_schd_data_item.ASTypeName == "NA"
                and net_schd_data_item.PXExchangeTypeName == "NA"
                and net_schd_data_item.PXTransactionTypeName == "NA"
                and net_schd_data_item.EnergyScheduleTypeName == "OA_REMC"
                and net_schd_data_item.EnergyScheduleSubTypeName == "OA_GNA"
                and net_schd_data_item.DataIntegrationTimingTypeName == "NA"
            )
            if cond:
                os_remc_list.append((date, net_schd_data_item))
    total_net_schd_amount.append((date, response_body.NetScheduleSummary.TotalNetSchdAmount))
    station_sched_amount.append((date, response_body.REMCDeclarationList.REMCDeclarationData.StationSchAmount if response_body.REMCDeclarationList else None))

In [ ]:
for date, item in os_remc_list:
    print(f"Date: {date}")
    print(item.__dict__)   

for date, item in as_emergency:
    print(f"Date: {date}")
    print(item.__dict__)

for date, item in as_shortfall:
    print(f"Date: {date}")
    print(item.__dict__)    

for date, amount in total_net_schd_amount:
    print(f"Date: {date}, TotalNetSchdAmount: {amount}")
for date, amount in station_sched_amount:
    print(f"Date: {date}, StationSchAmount: {amount}")

In [ ]:
import pandas as pd
from collections import defaultdict

# Converts whatever date string you stored into YYYY-MM-DD
def day_key(x):
    return pd.to_datetime(x).strftime("%Y-%m-%d")

# Build wide table: each column is a day, rows are index positions
# tuples_list format: [(date, value_or_object), ...]
# extractor returns either a list or a scalar from the second tuple item
def to_day_wide_df(tuples_list, extractor):
    by_day = defaultdict(list)

    for d, payload in tuples_list:
        day = day_key(d)
        vals = extractor(payload)

        if vals is None:
            continue
        if isinstance(vals, (list, tuple)):
            by_day[day].extend(vals)
        else:
            by_day[day].append(vals)

    ordered_days = sorted(by_day.keys())
    return pd.DataFrame({day: pd.Series(by_day[day]) for day in ordered_days})

# 1) os_remc_list -> use NetSchdAmount
df_os_remc = to_day_wide_df(os_remc_list, lambda item: item.NetSchdAmount)
df_os_remc.to_csv("os_remc_by_day.csv", index=True)

# 2) as_emergency -> use NetSchdAmount
df_as_emergency = to_day_wide_df(as_emergency, lambda item: item.NetSchdAmount)
df_as_emergency.to_csv("as_emergency_by_day.csv", index=True)

# 3) as_shortfall -> use NetSchdAmount
df_as_shortfall = to_day_wide_df(as_shortfall, lambda item: item.NetSchdAmount)
df_as_shortfall.to_csv("as_shortfall_by_day.csv", index=True)

# 4) total_net_schd_amount -> already a list/scalar amount
df_total_net = to_day_wide_df(total_net_schd_amount, lambda amount: amount)
df_total_net.to_csv("total_net_schd_by_day.csv", index=True)

# 5) station_sched_amount -> list/scalar or None
df_station = to_day_wide_df(station_sched_amount, lambda amount: amount)
df_station.to_csv("station_sch_by_day.csv", index=True)

print("Saved:")
print("- os_remc_by_day.csv")
print("- as_emergency_by_day.csv")
print("- as_shortfall_by_day.csv")
print("- total_net_schd_by_day.csv")
print("- station_sch_by_day.csv")